In [44]:
import numpy as np
from Izh_net import *
from net_preparation import *
from Var_Limb import *
from Rybak2002_final import *
import os
import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed
import dill
import sys
import logging

# Настройка логгирования
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Инициализация модели (должна быть в глобальной области)
def initialize_model():
    Flexor = SimpleAdaptedMuscle(w=0.5, N=50)
    Extensor = SimpleAdaptedMuscle(w=0.5, N=50)
    Flexor.tau_1 = 1/21
    Flexor.tau_c = 1/39
    Extensor.tau_1 = 1/21
    Extensor.tau_c = 1/39
    Pendulum = OneDOFLimb(
        q0=np.pi/2,
        w0=0.,
        b=0.01,
        a1=0.4,
        a2=0.05,
        m=0.3,
        l=0.3
    )
    Limb = Simple_Afferented_Limb(
        Limb=Pendulum,
        Flexor=Flexor,
        Extensor=Extensor
    )
    Qapp = np.zeros((12, 4))
    Qapp[0, 0] = Qapp[1, 1] = Qapp[6, 2] = Qapp[7, 3] = 1
    Rybak2002Net = Rybak_2002_network(
        input_size=4, 
        output_size=2, 
        afferent_size=6, 
        Qapp=Qapp, 
        exitatory_w=0.5, 
        inhibitory_w=-0.5
    )
    Rybak2002FullSystem = Var_Limb(Network=Rybak2002Net, Limb=Limb)
    Rybak2002FullSystem.set_afferents_by_names('Ia_Flex', 'Ia_IN_Flex', 10)
    Rybak2002FullSystem.set_afferents_by_names('Ia_Ext', 'Ia_IN_Ext', 10)
    
    return Rybak2002FullSystem

# Генератор импульсного тока
def make_pulse_generator(params):
    period = [
        params["pulse_period_ch0"],
        params["pulse_period_ch1"],
        params["pulse_period_ch2"],
        params["pulse_period_ch3"]
    ]
    durations = [
        params["pulse_duration_ch0"],
        params["pulse_duration_ch1"],
        params["pulse_duration_ch2"],
        params["pulse_duration_ch3"]
    ]
    amplitudes = [
        params["amplitude_ch0"],
        params["amplitude_ch1"],
        params["amplitude_ch2"],
        params["amplitude_ch3"]
    ]
    phases = [
        params["phase_ch0"],
        params["phase_ch1"],
        params["phase_ch2"],
        params["phase_ch3"]
    ]
    base_currents = [
        params["base_current_I1"],
        params["base_current_I2"],
        params["base_current_I3"],
        params["base_current_I4"]
    ]
    noise_percent = params["noise_percent"]
    
    def Iapp(t):
        I = np.zeros(4)
        for i in range(4):
            t_phase = (t - phases[i] * period[i]) % period[i]
            pulse_value = amplitudes[i] if t_phase < durations[i] else 0.0
            noise = np.random.normal(0, base_currents[i] * noise_percent)
            I[i] = base_currents[i] + pulse_value + noise
        return I
    
    return Iapp

# Основная функция выполнения симуляции
def run_simulation_task(params):
    try:
        logger.info(f"Запуск симуляции для комбинации {params['combination_id']}")
        model = initialize_model()
        Iapp = make_pulse_generator(params)
        
        scale = 5
        Tmax = 10000
        T = np.linspace(0, Tmax, scale*Tmax)
        dt = T[1] - T[0]
        N = 12
        
        V_curr = np.zeros((len(T), N))
        U_curr = np.zeros((len(T), N))
        F_flex_curr = np.zeros(len(T))
        F_ext_curr = np.zeros(len(T))
        Afferents_curr = np.zeros((len(T), 6))
        Q_curr = np.zeros(len(T))
        W_curr = np.zeros(len(T))
        
        for i, t in enumerate(T):
            V_curr[i] = model.V
            U_curr[i] = model.U
            F_flex_curr[i] = model.F_flex
            F_ext_curr[i] = model.F_ext
            Afferents_curr[i] = model.Limb.output
            Q_curr[i] = model.q
            W_curr[i] = model.w
            model.step(dt=dt, Iapp=Iapp(t))
        
        start_index = int(len(T) * 0.5)
        T_trimmed = T[start_index:]
        V_trimmed = V_curr[start_index:]
        U_trimmed = U_curr[start_index:]
        F_flex_trimmed = F_flex_curr[start_index:]
        F_ext_trimmed = F_ext_curr[start_index:]
        Afferents_trimmed = Afferents_curr[start_index:]
        Q_trimmed = Q_curr[start_index:]
        W_trimmed = W_curr[start_index:]

        # Округление значений до сотых для использования в названии файла
        period = [round(params[f"pulse_period_ch{i}"], 2) for i in range(4)]
        durations = [round(params[f"pulse_duration_ch{i}"], 2) for i in range(4)]
        amplitudes = [round(params[f"amplitude_ch{i}"], 2) for i in range(4)]
        phases = [round(params[f"phase_ch{i}"], 2) for i in range(4)]

        # Форматирование строк с округленными значениями
        var_str_period = "_".join([f"{x:.2f}" for x in period])
        var_str_durations = "_".join([f"{x:.2f}" for x in durations])
        var_str_amplitudes = "_".join([f"{x:.2f}" for x in amplitudes])
        var_str_phases = "_".join([f"{x:.2f}" for x in phases])
        
        output_dir = "experiment_data_pulses111"
        os.makedirs(output_dir, exist_ok=True)
        data_filename = f"comb_{params['combination_id']}_per_{var_str_period}_dur_{var_str_durations}_amp_{var_str_amplitudes}_ph{var_str_phases}.npz"
        
        np.savez_compressed(
            os.path.join(output_dir, data_filename),
            T=T_trimmed,
            V=V_trimmed,
            U=U_trimmed,
            F_flex=F_flex_trimmed,
            F_ext=F_ext_trimmed,
            Afferents=Afferents_trimmed,
            Q=Q_trimmed,
            W=W_trimmed,
            dt=dt,
            Tmax=Tmax
        )
        
        logger.info(f"✅ Успешно завершена комбинация {params['combination_id']}")
        return {
            "combination_id": params['combination_id'],
            "data_file": data_filename,
            "status": "success"
        }
    except Exception as e:
        logger.error(f"❌ Ошибка в комбинации {params['combination_id']}: {str(e)}")
        return {
            "combination_id": params['combination_id'],
            "status": "error",
            "error": str(e)
        }

# Основная функция управления
def main():
    print("="*50)
    print("🧪 Программа моделирования импульсных воздействий")
    print("="*50)
    
    config_file = "experiment_configs/pulse_parameters.csv"
    if not os.path.exists(config_file):
        print(f"❌ Файл конфигурации не найден: {config_file}")
        print("ℹ️ Сначала запустите config_generator_pulses.py")
        return
    
    df = pd.read_csv(config_file)
    print(f"📋 Загружено {len(df)} конфигураций")
    
    print("\n🔧 Режимы работы:")
    print("1. Запуск по ID комбинации")
    print("2. Запуск по диапазону ID")
    print("3. Запуск всех комбинаций")
    
    choice = input("👉 Выберите режим (1-3): ")
    selected_params_list = []
    
    if choice == "1":
        comb_id = int(input("🔢 Введите ID комбинации: "))
        if comb_id not in df["combination_id"].values:
            print(f"❌ Комбинация {comb_id} не найдена")
            return
        selected_params_list = [df[df["combination_id"] == comb_id].iloc[0].to_dict()]
        
    elif choice == "2":
        start_id = int(input("🔢 Введите начальный ID: "))
        end_id = int(input("🔢 Введите конечный ID: "))
        selected = df[(df["combination_id"] >= start_id) & (df["combination_id"] <= end_id)]
        selected_params_list = [row.to_dict() for _, row in selected.iterrows()]
            
    elif choice == "3":
        selected_params_list = [row.to_dict() for _, row in df.iterrows()]
            
    else:
        print("❌ Некорректный выбор")
        return
    
    n_jobs = int(input("Введите количество ядер (n_jobs, -1 для всех): ") or -1)
    
    # Настройка параллельного выполнения
    results = Parallel(n_jobs=n_jobs, verbose=50, backend='loky')(
        delayed(run_simulation_task)(params) 
        for params in tqdm(selected_params_list, desc="Подготовка задач")
    )
    
    # Сохранение результатов
    if results:
        results_df = pd.DataFrame(results)
        results_df.to_csv("experiment_results_pulses111.csv", index=False)
        print("✅ Результаты сохранены в experiment_results_pulses111.csv")
        
        # Статистика выполнения
        success_count = results_df[results_df['status'] == 'success'].shape[0]
        error_count = results_df[results_df['status'] == 'error'].shape[0]
        print(f"\n📊 Статистика выполнения:")
        print(f"  Успешно: {success_count}")
        print(f"  С ошибками: {error_count}")
        
        if error_count > 0:
            print("\n❌ Ошибочные комбинации:")
            print(results_df[results_df['status'] == 'error'][['combination_id', 'error']])

# Точка входа с обработкой Windows-специфичных проблем
if __name__ == "__main__":
    # Решение проблем Windows
    if sys.platform.startswith('win'):
        from multiprocessing import freeze_support
        freeze_support()
        
        # Улучшенная обработка сериализации
        dill.settings['recurse'] = True
    
    try:
        main()
    except KeyboardInterrupt:
        print("\n🚫 Программа прервана пользователем")
    except Exception as e:
        print(f"\n🔥 Критическая ошибка: {str(e)}")
        import traceback
        traceback.print_exc()

🧪 Программа моделирования импульсных воздействий
📋 Загружено 54756 конфигураций

🔧 Режимы работы:
1. Запуск по ID комбинации
2. Запуск по диапазону ID
3. Запуск всех комбинаций


👉 Выберите режим (1-3):  3
Введите количество ядер (n_jobs, -1 для всех):  10











Подготовка задач:   0%|                                                                      | 0/54756 [00:00<?, ?it/s]










[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.


Подготовка задач:   0%|                                                             | 10/54756 [00:00<17:11, 53.10it/s]








Подготовка задач:   0%|                                                           | 20/54756 [00:04<3:42:37,  4.10it/s]

...










Подготовка задач: 100%|██████████████████████████████████████████████████████▉| 54740/54756 [35:16:31<00:34,  2.16s/it]

[Parallel(n_jobs=10)]: Done 54721 tasks      | elapsed: 2116.5min
[Parallel(n_jobs=10)]: Done 54722 tasks      | elapsed: 2116.6min
[Parallel(n_jobs=10)]: Done 54723 tasks      | elapsed: 2116.6min
[Parallel(n_jobs=10)]: Done 54724 tasks      | elapsed: 2116.6min
[Parallel(n_jobs=10)]: Done 54725 tasks      | elapsed: 2116.8min
[Parallel(n_jobs=10)]: Done 54726 tasks      | elapsed: 2116.8min
[Parallel(n_jobs=10)]: Done 54727 tasks      | elapsed: 2116.8min
[Parallel(n_jobs=10)]: Done 54728 tasks      | elapsed: 2116.8min
[Parallel(n_jobs=10)]: Done 54729 tasks      | elapsed: 2116.8min
[Parallel(n_jobs=10)]: Done 54730 tasks      | elapsed: 2116.9min











Подготовка задач: 100%|███████████████████████████████████████████████████████| 54756/54756 [35:16:53<00:00,  2.32s/it]

[Parallel(n_jobs=10)]: Done 54731 tasks      | elapsed: 2116.9min


[Parallel(n_jobs=10)]: Done 54732 tasks      | elapsed: 2116.9min
[Parallel(n_jobs=10)]: Done 54733 tasks      | elapsed: 2117.0min
[Parallel(n_jobs=10)]: Done 54734 tasks      | elapsed: 2117.0min
[Parallel(n_jobs=10)]: Done 54735 tasks      | elapsed: 2117.1min
[Parallel(n_jobs=10)]: Done 54736 tasks      | elapsed: 2117.1min
[Parallel(n_jobs=10)]: Done 54737 tasks      | elapsed: 2117.1min
[Parallel(n_jobs=10)]: Done 54756 out of 54756 | elapsed: 2117.7min finished
✅ Результаты сохранены в experiment_results_pulses111.csv

📊 Статистика выполнения:
  Успешно: 54756
  С ошибками: 0
